# Lab 1 — Gemini on Vertex AI (OpenAI-compatible endpoint + ADC)

This notebook mirrors `1_lab1.ipynb`, but calls **Gemini on Google Vertex AI** using the **OpenAI Python SDK** and **Application Default Credentials (ADC)** instead of an API key.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Vertex AI setup</h2>
            <span style="color:#ff7800;">
            Before running this notebook:<br/>
            1. Enable the Vertex AI API in your Google Cloud project.<br/>
            2. Authenticate locally with ADC: <code>gcloud auth application-default login</code><br/>
            3. Set your default project: <code>gcloud config set project YOUR_PROJECT_ID</code><br/>
            4. Optionally add <code>GOOGLE_CLOUD_PROJECT</code> and <code>GOOGLE_CLOUD_LOCATION</code> to your <code>.env</code> file in the project root.<br/>
            See the <a href="https://cloud.google.com/vertex-ai/generative-ai/docs/migrate/openai/auth-and-credentials">Vertex OpenAI auth docs</a> for details.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Same lab flow, different provider</h2>
            <span style="color:#00bfff;">We still use the familiar OpenAI <code>messages</code> format and <code>chat.completions.create()</code> calls. Vertex exposes an OpenAI-compatible endpoint; we pass a short-lived OAuth token from ADC as <code>api_key</code> instead of a static key.<br/><br/>
            Model IDs on Vertex use the <code>google/</code> prefix, for example <code>google/gemini-2.5-flash</code>.
            </span>
        </td>
    </tr>
</table>

### New to Notebooks like this one? Head over to the guides folder!

1. Click where it says "Select Kernel" near the top right, and select `.venv (Python 3.12.x)` or similar.
2. Click in each cell below and press Shift+Enter to run.
3. Enjoy!

In [ ]:
from dotenv import load_dotenv

In [ ]:
# Load optional .env values (project id, location) — no API key needed for Vertex ADC
load_dotenv(override=True)

### Project and region

Set `GOOGLE_CLOUD_PROJECT` and `GOOGLE_CLOUD_LOCATION` in your repo-root `.env`, or rely on `gcloud config` / ADC defaults. Use `global` for Gemini 3.x preview models; regional endpoints (e.g. `us-central1`) work for most stable models.

In [ ]:
import os
import subprocess

from google.auth import default
import google.auth.transport.requests
from openai import OpenAI


def _gcloud_project() -> str | None:
    try:
        return subprocess.check_output(
            ["gcloud", "config", "get-value", "project"],
            text=True,
        ).strip() or None
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


credentials, adc_project = default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
credentials.refresh(google.auth.transport.requests.Request())

project_id = (
    os.getenv("GOOGLE_CLOUD_PROJECT")
    or os.getenv("GCP_PROJECT")
    or adc_project
    or _gcloud_project()
)
location = os.getenv("GOOGLE_CLOUD_LOCATION", "global")

if project_id:
    print("Vertex project configured (from .env, ADC, or gcloud config)")
    print(f"Using Vertex location: {location}")
    print("ADC token obtained (expires in ~1 hour)")
else:
    print("Could not determine Google Cloud project id.")
    print("Set GOOGLE_CLOUD_PROJECT in .env or run: gcloud config set project YOUR_PROJECT_ID")

In [ ]:
# Create an OpenAI client pointed at Vertex's OpenAI-compatible endpoint.
# Vertex uses Google OAuth tokens instead of static API keys.

def vertex_openai_base_url(project: str, region: str) -> str:
    if region == "global":
        host = "aiplatform.googleapis.com"
    else:
        host = f"{region}-aiplatform.googleapis.com"
    return (
        f"https://{host}/v1/projects/{project}/locations/{region}/endpoints/openapi"
    )


VERTEX_BASE_URL = vertex_openai_base_url(project_id, location)

vertex = OpenAI(base_url=VERTEX_BASE_URL, api_key=credentials.token)

# Vertex model ids use the google/ prefix (override via .env if needed)
MODEL_FAST = os.getenv("GEMINI_MODEL_FAST", "google/gemini-2.5-flash-lite")
MODEL_BALANCED = os.getenv("GEMINI_MODEL_BALANCED", "google/gemini-2.5-flash")
MODEL_STRONG = os.getenv("GEMINI_MODEL_STRONG", "google/gemini-2.5-pro")

print("Vertex OpenAI-compatible client ready")

In [ ]:
# Create a list of messages in the familiar OpenAI format

messages = [{"role": "user", "content": "Tell me a fun fact"}]

In [ ]:
messages

In [ ]:
# First call — fast, inexpensive Gemini model on Vertex

response = vertex.chat.completions.create(model=MODEL_FAST, messages=messages)
print(response.choices[0].message.content)

In [ ]:
# Ask the model to propose a hard question

question = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."
messages = [{"role": "user", "content": question}]

In [ ]:
response = vertex.chat.completions.create(model=MODEL_BALANCED, messages=messages)
question = response.choices[0].message.content
print(question)

In [ ]:
messages = [{"role": "user", "content": question}]
messages

In [ ]:
response = vertex.chat.completions.create(model=MODEL_BALANCED, messages=messages)
answer = response.choices[0].message.content
print(answer)

In [ ]:
from IPython.display import Markdown, display

display(Markdown(answer))

In [ ]:
message = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""

print(message)

In [ ]:
messages = [{"role": "user", "content": message}]
response = vertex.chat.completions.create(model=MODEL_STRONG, messages=messages)
print(response.choices[0].message.content)

# Congratulations!

You ran the same Agentic AI flow as Lab 1, but through **Gemini on Vertex AI** with **ADC** and the **OpenAI-compatible endpoint**.

If you get auth errors after ~1 hour, re-run the ADC setup cell to refresh the token.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try this commercial application:<br/>
            First ask the LLM to pick a business area that might be worth exploring for an Agentic AI opportunity.<br/>
            Then ask the LLM to present a pain-point in that industry - something challenging that might be ripe for an Agentic solution.<br/>
            Finally have a third LLM call propose the Agentic AI solution.<br/>
            Use <code>vertex.chat.completions.create(...)</code> with the same model constants as above.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# First create the messages:

messages = [{"role": "user", "content": "Something here"}]

# Then make the first call:

response =

# Then read the business idea:

business_area = response.

# And repeat! In the next message, include the business area within the message